# PyHealth Tutorial 04: Choosing a model: from logistic regression to XGBoost

**What you will learn**
- What the main PyHealth models assume about the data, and when to pick each
- How to compare several models fairly on the same patients
- How to add a gradient-boosted tree baseline (XGBoost) to a PyHealth benchmark

**Time:** about 20 minutes on a free Colab CPU runtime.  
**Before this:** Tutorial 03.

All data used here is synthetic or public, so every cell runs without credentials.

## Setup

These tutorials use features from the upcoming PyHealth 2.1 release, so for now we install the latest version straight from GitHub.

**On Colab this takes two clicks of Runtime > Run all.** The first run installs PyHealth and restarts the runtime, because Colab has already loaded older versions of some packages (such as numpy). Colab then says "Your session crashed": that is expected. Choose Run all again and the tutorial runs through. On your own machine, install PyHealth once (`pip install "pyhealth[xgboost]"`) and this cell does nothing.

In [ ]:
# First run: installs PyHealth, then restarts the runtime once so Python
# loads the new package versions. Then choose Runtime > Run all again;
# this cell will see PyHealth is installed and skip the install.
import importlib.util
import os
import subprocess
import sys

PYHEALTH = "pyhealth[xgboost] @ git+https://github.com/sunlabuiuc/PyHealth.git"

if importlib.util.find_spec("pyhealth") is None:
    print("Installing PyHealth (1-2 minutes)...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", PYHEALTH],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print(result.stdout[-5000:], result.stderr[-5000:])
        raise RuntimeError("Installing PyHealth failed; see the log above.")
    print("Installed. Restarting the runtime now; then choose Runtime > Run all again.")
    os.kill(os.getpid(), 9)  # Colab restarts the runtime automatically
else:
    import logging
    import pyhealth
    logging.getLogger("pyhealth").propagate = False  # print each log line once
    print("PyHealth is installed and ready.")

## The models at a glance

All PyHealth models take the dataset in their constructor and read the
input and output schemas from it, so swapping models is a one-line
change. The main ones for coded EHR data:

| Model | Idea | Good first choice when |
|---|---|---|
| `LogisticRegression` | Embeds each code, sums them, one linear layer | You want a transparent, hard-to-beat baseline |
| `MLP` | Same pooled embeddings, then hidden layers | Features interact, but order does not matter |
| `RNN` | Reads each field as a sequence (GRU or LSTM) | Order within a field carries signal |
| `Transformer` | Self-attention over each field | Longer histories, long-range interactions |
| `RETAIN` | Two attention layers over visits | You need visit- and code-level attention weights |
| `XGBoostModel` | Gradient-boosted trees on code counts or tabular features | Tabular or bag-of-codes data; the standard strong baseline |

On tabular and bag-of-codes data, gradient-boosted trees are often as
good as neural models and much faster to train, so a benchmark without
them can overstate what a neural model adds.

## The task: spotting heart failure from the medication list

We need a question the synthetic data can answer, so we use a
*phenotyping* task: does this admission include a heart-failure
diagnosis (ICD-9 428.x), judged from the drugs and procedures recorded
in it? Phenotyping models like this help find patients with a
condition when diagnosis codes are missing or unreliable, for example
when building a study cohort. About 1 in 5 admissions qualifies.

The task is a few lines (Tutorial 02 explains each part). Drugs are
kept by name so the results stay readable.

In [ ]:
from collections import defaultdict

from pyhealth.tasks import BaseTask


class HeartFailureFromMedications(BaseTask):
    """Does an admission include a heart-failure diagnosis (ICD-9 428.x)?"""

    task_name = "HeartFailureFromMedications"
    input_schema = {"drugs": "sequence", "procedures": "sequence"}
    output_schema = {"heart_failure": "binary"}

    def __call__(self, patient):
        # Fetch each event type once, then group by admission: much
        # faster than one filtered query per admission.
        by_admission = defaultdict(lambda: {"drugs": [], "diagnoses": [], "procedures": []})
        for e in patient.get_events(event_type="prescriptions"):
            if e.drug:
                by_admission[e.hadm_id]["drugs"].append(e.drug)
        for e in patient.get_events(event_type="diagnoses_icd"):
            by_admission[e.hadm_id]["diagnoses"].append(e.icd9_code)
        for e in patient.get_events(event_type="procedures_icd"):
            by_admission[e.hadm_id]["procedures"].append(e.icd9_code)

        samples = []
        for hadm_id, codes in by_admission.items():
            if not codes["drugs"] or not codes["diagnoses"]:
                continue
            samples.append({
                "patient_id": patient.patient_id,
                "visit_id": hadm_id,
                "drugs": codes["drugs"],
                "procedures": codes["procedures"] or ["none"],
                "heart_failure": int(any(c.startswith("428") for c in codes["diagnoses"])),
            })
        return samples

## One split for every model

A fair comparison uses the same patients for every model, so we build
the samples once, split by patient (Tutorial 03), and reuse the three
parts.

In [ ]:
from pyhealth.datasets import MIMIC3Dataset, PatientSplit, get_dataloader

dataset = MIMIC3Dataset(
    root="https://storage.googleapis.com/pyhealth/Synthetic_MIMIC-III",
    tables=["diagnoses_icd", "procedures_icd", "prescriptions"],
    cache_dir="pyhealth_cache",
)
train_ds, val_ds, test_ds = dataset.set_task(
    HeartFailureFromMedications(), split=PatientSplit((0.7, 0.1, 0.2), seed=7)
)
print(f"train {len(train_ds)}  val {len(val_ds)}  test {len(test_ds)}")

val_loader = get_dataloader(val_ds, batch_size=512)
test_loader = get_dataloader(test_ds, batch_size=512)

## Neural models

One helper trains any model the same way: same optimizer, same
maximum epochs, the best epoch picked on validation PR-AUC, and the
same test set. Only the model changes.

Sequence models are slow on a CPU with 35,000 training admissions, so
on a CPU runtime we train the three fastest; on a GPU runtime
(Runtime > Change runtime type) all five run in a few minutes.

In [ ]:
import time

import numpy as np
import torch
from pyhealth.models import MLP, RETAIN, RNN, LogisticRegression, Transformer
from pyhealth.trainer import Trainer

METRICS = ["pr_auc", "roc_auc"]
results = {}


def fit_and_test(name, model_class, **kwargs):
    torch.manual_seed(0)
    model = model_class(dataset=train_ds, **kwargs)
    trainer = Trainer(model=model, metrics=METRICS, output_path="runs", exp_name=name)
    start = time.time()
    trainer.train(
        train_dataloader=get_dataloader(train_ds, batch_size=256, shuffle=True),
        val_dataloader=val_loader,
        epochs=6,
        monitor="pr_auc",
        patience=2,
    )
    scores = trainer.evaluate(test_loader)
    n_params = sum(p.numel() for p in model.parameters())
    results[name] = {**scores, "seconds": time.time() - start, "parameters": n_params}
    return trainer


models = [("LogisticRegression", LogisticRegression), ("MLP", MLP), ("RNN", RNN)]
if torch.cuda.is_available():
    models += [("Transformer", Transformer), ("RETAIN", RETAIN)]
else:
    print("CPU runtime: skipping Transformer and RETAIN (about 10 minutes each on CPU). "
          "Switch to a GPU runtime to include them.")
for name, cls in models:
    fit_and_test(name, cls)

## Adding XGBoost

Gradient-boosted trees do not train with `Trainer.train`: they are fit
once on the full training set with `model.fit(...)`. After that they
work with the same `Trainer.evaluate`, metrics and checkpoints as
every other model.

Trees need fixed-width numeric inputs. Code lists vary in length, so
`bag_of_codes=True` turns each field into per-sample code counts over
the vocabulary (one column per code). `early_stopping_rounds` uses the
validation set to choose the number of trees.

Use unshuffled loaders for `fit`: the order of rows affects row
subsampling, so a fixed order makes the fit reproducible.

In [ ]:
from pyhealth.models import XGBoostModel

start = time.time()
xgb = XGBoostModel(
    train_ds,
    bag_of_codes=True,
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=0,
    early_stopping_rounds=50,
    eval_metric="aucpr",
)
xgb.fit(get_dataloader(train_ds, batch_size=1024), val_loader)
scores = Trainer(model=xgb, metrics=METRICS, enable_logging=False).evaluate(test_loader)
results["XGBoost"] = {**scores, "seconds": time.time() - start,
                      "parameters": xgb.estimators_[0].best_iteration + 1}
print(f"{len(xgb.feature_names)} columns, e.g. {xgb.feature_names[:3]}")
print(f"best number of trees: {xgb.estimators_[0].best_iteration + 1}")

## Results

In [ ]:
prevalence = float(np.mean([int(test_ds[i]["heart_failure"]) for i in range(len(test_ds))]))
print(f"test prevalence (PR-AUC of a random guess): {prevalence:.3f}\n")
print(f"{'model':20s} {'PR-AUC':>7s} {'ROC-AUC':>8s} {'seconds':>8s} {'size':>10s}")
for name, r in sorted(results.items(), key=lambda kv: -kv[1]["pr_auc"]):
    size = f"{r['parameters']} trees" if name == "XGBoost" else f"{r['parameters']:,}"
    print(f"{name:20s} {r['pr_auc']:7.3f} {r['roc_auc']:8.3f} {r['seconds']:8.1f} {size:>10s}")

### Reading the table

- Compare every model with the random-guess line first: PR-AUC is only
  meaningful relative to the prevalence.
- Look at cost as well as score: training time and model size differ
  by orders of magnitude for similar accuracy.
- Small gaps can be noise. Before
  declaring a winner, compute bootstrap intervals of the paired
  difference (Tutorial 03, `paired_bootstrap_diff`).
- Neural model scores also move with the random seed. Report the mean
  and spread over several seeds rather than one run.

## Rules of thumb

1. Start with `LogisticRegression` and `XGBoostModel`. They are fast
   and often strong, and every other model has to beat them.
2. Move to `RNN`, `Transformer` or `RETAIN` when order or long
   histories plausibly matter, and check the gain with intervals.
3. Keep the split, the metric used to pick the epoch, and the test set
   identical across models.

## Summary

- Every model reads the schemas, so swapping models is one line.
- `XGBoostModel` fits with `model.fit(train, val)` and then works with
  the standard `Trainer` evaluation; `bag_of_codes=True` handles code
  lists.
- Compare on the same patients, report intervals, and include strong
  simple baselines.

## Where to go next

**Next:** Tutorial 05, *Interpreting predictions*.

The whole series:

00. Quickstart: your first clinical prediction model
01. Datasets: patients, events and your own data
02. Tasks and processors: from patients to model-ready samples
03. Training and evaluating models properly
04. Choosing a model: from logistic regression to XGBoost
05. Interpreting predictions
06. Medical codes: lookups, mappings and groupers
07. Clinical text: classification and medical coding
08. Contributing a dataset or task to PyHealth

Questions or problems? Open an issue at https://github.com/sunlabuiuc/PyHealth/issues. If PyHealth is useful to you, a star on GitHub helps the project.